# Notebook 25 — Downstream recomputation after the A13 parameter correction

**Author:** Sunita Sahu · **Revision item:** A13 (downstream recomputation)

Notebook 24 corrected the parameter attribution (IDEFICS-9B cross-attention layers, Janus-Pro generation modules, ViLT pooler; double counts in InternVL3, Chameleon, Qwen2-VL and Qwen2.5-VL). This notebook propagates those corrections to every downstream number:

| Section | What is recomputed | Validation before any change |
|---|---|---|
| 1–2 | Table 3, analytic forward compute | all 20 published rows reproduced to printed precision |
| 3 | Table 7 + Spearman ρ(V%, C_V%), n = 19 | published ρ and Fisher interval reproduced |
| 4 | Table 8, 28 marginal + 28 year-partial tests | every published ρ, partial ρ and exact (n ≤ 8) p reproduced |
| 5 | Section 5.8 composite score, score/TFLOP, Pareto sets | published Table 9/10 reproduced |

**Only A13-affected inputs change.** Other known issues found on the way (BLIP-2 encoder layer count, lm-head convention, Idefics2 trainable tie) are reported, not fixed — they belong to other reviewer items.

---
## Setup

In [1]:
import sys, json, math
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
CODE_ACTIVE = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
sys.path.insert(0, str(CODE_ACTIVE))
MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
ARCHIVE_DIR = CODE_ACTIVE / '_archive' / 'pre_A13_measurements'

from vlm_audit.analytic_compute import ComputeInputs, forward_cost, fisher_interval
from vlm_audit.correlation_grid import spearman_with_partial, spearman_permutation_test, benjamini_hochberg
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 60)

def load_record(directory, file_stem):
    """Return the components dict of one per-model JSON record."""
    return json.loads((directory / f'{file_stem}.json').read_text(encoding='utf-8'))['components']

---
## 1. Analytic compute inputs (protocol of Section 3.2)

Layer counts, widths and token counts below reproduce the published Table 3. Two published conventions are kept deliberately (they are outside A13 and are flagged for reviewer item A5): BLIP-2/InstructBLIP's EVA encoder is costed at 40 layers (the released config has 39), and the LM head is inside P_language for the four models whose head sits inside the language module (InternVL3, Molmo, DeepSeek-VL2, Janus-Pro) but outside it elsewhere. DeepSeek-VL2's MoE language model is costed at 2.36 B activated parameters.

In [2]:
# name: record stem, design, encoder (layers, tokens, width), fusion tokens, language (layers, width),
#       visual tokens in LLM, cross-attention layers, encoder component, language components
COMPUTE_SPECS = {
    'CLIP-ViT-L/14': ('openai_clip-vit-large-patch14', 'contrastive', (24, 257, 1024), 0, (12, 768), 0, 0, 'vision', ['language']),
    'SigLIP-L/16': ('google_siglip-large-patch16-384', 'contrastive', (24, 576, 1024), 0, (24, 1024), 0, 0, 'vision', ['language']),
    'SigLIP2-So400m/14': ('google_siglip2-so400m-patch14-384', 'contrastive', (27, 729, 1152), 0, (27, 1152), 0, 0, 'vision', ['language']),
    'BLIP-large': ('Salesforce_blip-image-captioning-large', 'contrastive', (24, 577, 1024), 0, (12, 768), 0, 0, 'vision', ['language']),
    'BLIP-2-OPT-2.7B': ('Salesforce_blip2-opt-2.7b', 'inserted', (40, 257, 1408), 32, (32, 2560), 32, 0, 'vision', ['language']),
    'InstructBLIP-Vicuna-7B': ('Salesforce_instructblip-vicuna-7b', 'inserted', (40, 257, 1408), 32, (32, 4096), 32, 0, 'vision', ['language']),
    'LLaVA-1.5-7B': ('llava-hf_llava-1.5-7b-hf', 'inserted', (24, 577, 1024), 576, (32, 4096), 576, 0, 'vision', ['language']),
    'LLaVA-NeXT-7B': ('llava-hf_llava-v1.6-mistral-7b-hf', 'inserted', (24, 577, 1024), 576, (32, 4096), 576, 0, 'vision', ['language']),
    'Molmo-7B-D': ('allenai_Molmo-7B-D-0924', 'inserted', (24, 577, 1024), 0, (28, 3584), 576, 0, 'vision', ['language']),
    'IDEFICS-9B': ('HuggingFaceM4_idefics-9b', 'cross_attention', (32, 257, 1280), 64, (32, 4096), 64, 8, 'vision', ['language']),
    'Llama-3.2-11B-Vision': ('meta-llama_Llama-3.2-11B-Vision-Instruct', 'cross_attention', (40, 1601, 1280), 1601, (32, 4096), 1601, 8, 'vision', ['language']),
    'Idefics2-8B': ('HuggingFaceM4_idefics2-8b', 'inserted', (27, 729, 1152), 64, (32, 4096), 64, 0, 'vision', ['language']),
    'ViLT-B/32': ('dandelin_vilt-b32-finetuned-vqa', 'shared_stream', None, 0, (12, 768), 144, 0, None, ['language']),
    'Qwen2-VL-7B': ('Qwen_Qwen2-VL-7B-Instruct', 'inserted', (32, 1024, 1280), 256, (28, 3584), 256, 0, 'vision', ['language']),
    'Qwen2.5-VL-7B': ('Qwen_Qwen2.5-VL-7B-Instruct', 'inserted', (32, 1024, 1280), 256, (28, 3584), 256, 0, 'vision', ['language']),
    'InternVL3-8B': ('OpenGVLab_InternVL3-8B', 'inserted', (24, 1025, 1024), 256, (28, 3584), 256, 0, 'vision', ['language', 'head']),
    'Pixtral-12B': ('mistral-community_pixtral-12b', 'inserted', (24, 4096, 1024), 4096, (40, 5120), 4096, 0, 'vision', ['language']),
    'DeepSeek-VL2-Small': ('deepseek-ai_deepseek-vl2-small', 'inserted', (27, 729, 1152), 729, (27, 2048), 729, 0, 'vision', ['ACTIVATED:2.36e9']),
    'Chameleon-7B': ('facebook_chameleon-7b', 'shared_stream', None, 0, (32, 4096), 1024, 0, None, ['fused_transformer', 'embeddings']),
    'Janus-Pro-7B': ('deepseek-ai_Janus-Pro-7B', 'inserted', (24, 576, 1024), 576, (30, 4096), 576, 0, 'vision_und', ['language']),
}

def build_inputs(name, record_directory):
    """ComputeInputs for one model, taking parameter counts from the given record directory."""
    stem, design, encoder, fusion_tokens, language, visual_tokens, cross_layers, encoder_key, language_keys = COMPUTE_SPECS[name]
    components = load_record(record_directory, stem)
    def count(key):
        return (components.get(key) or {}).get('params_total') or 0
    # Pre-A13 records stored overlapping modules (the head inside language for InternVL3, the
    # embeddings inside the fused transformer for Chameleon), so only the first key is read there.
    keys_to_sum = language_keys if record_directory == MEASUREMENT_DIR else language_keys[:1]
    if language_keys[0].startswith('ACTIVATED:'):
        language_parameters = float(language_keys[0].split(':')[1])
    else:
        language_parameters = sum(count(key) for key in keys_to_sum)
    return ComputeInputs(
        model=name, design=design,
        encoder_parameters=count(encoder_key) if encoder_key else 0,
        encoder_layers=encoder[0] if encoder else 0, encoder_tokens=encoder[1] if encoder else 0,
        encoder_width=encoder[2] if encoder else 0,
        fusion_parameters=count('fusion') if fusion_tokens else 0, fusion_output_tokens=fusion_tokens,
        language_parameters=language_parameters, language_layers=language[0], language_width=language[1],
        visual_tokens_in_language=visual_tokens, cross_attention_layers=cross_layers)

published_inputs = {name: build_inputs(name, ARCHIVE_DIR) for name in COMPUTE_SPECS}
revised_inputs = {name: build_inputs(name, MEASUREMENT_DIR) for name in COMPUTE_SPECS}
inputs_table = pd.DataFrame([inputs.to_dict() for inputs in revised_inputs.values()])
inputs_table.to_csv(MEASUREMENT_DIR / 'analytic_compute_inputs.csv', index=False)
inputs_table[['model','design','encoder_parameters','encoder_layers','encoder_tokens','encoder_width','fusion_parameters',
              'fusion_output_tokens','language_parameters','language_layers','language_width','visual_tokens_in_language','cross_attention_layers']]

,model,design,encoder_parameters,encoder_layers,encoder_tokens,encoder_width,fusion_parameters,fusion_output_tokens,language_parameters,language_layers,language_width,visual_tokens_in_language,cross_attention_layers
0,CLIP-ViT-L/14,contrastive,303179776,24,257,1024,0,0,1.230605e+08,12,768,0,0
1,SigLIP-L/16,contrastive,316283904,24,576,1024,0,0,3.361946e+08,24,1024,0,0
2,SigLIP2-So400m/14,contrastive,428225600,27,729,1152,0,0,7.077829e+08,27,1152,0,0
3,BLIP-large,contrastive,303690752,24,577,1024,0,0,1.425997e+08,12,768,0,0
4,BLIP-2-OPT-2.7B,inserted,985952256,40,257,1408,105162240,32,2.651679e+09,32,2560,32,0
5,InstructBLIP-Vicuna-7B,inserted,985952256,40,257,1408,185684736,32,6.738940e+09,32,4096,32,0
6,LLaVA-1.5-7B,inserted,303507456,24,577,1024,20979712,576,6.607606e+09,32,4096,576,0
7,LLaVA-NeXT-7B,inserted,303507456,24,577,1024,20983808,576,7.110922e+09,32,4096,576,0
8,Molmo-7B-D,inserted,404950016,24,577,1024,0,0,7.616075e+09,28,3584,576,0
9,IDEFICS-9B,cross_attention,630766080,32,257,1280,125961984,64,8.041871e+09,32,4096,64,8


---
## 2. Table 3 — reproduction, then A13 revision

In [3]:
PUBLISHED_TABLE3 = {  # encoder, fusion, LLM-visual, LLM-text, total, C_V%, vision-attributable%
    'CLIP-ViT-L/14': (162.3, 0.0, 0.0, 7.9, 170.2, 95.4, 95.4), 'SigLIP-L/16': (397.0, 0.0, 0.0, 21.6, 418.6, 94.8, 94.8),
    'SigLIP2-So400m/14': (690.5, 0.0, 0.0, 45.4, 735.9, 93.8, 93.8), 'BLIP-large': (383.2, 0.0, 0.0, 9.2, 392.4, 97.7, 97.7),
    'BLIP-2-OPT-2.7B': (521.7, 6.73, 170.4, 170.4, 869.1, 60.0, 80.4), 'InstructBLIP-Vicuna-7B': (521.7, 11.88, 432.4, 432.4, 1398.3, 37.3, 69.1),
    'LLaVA-1.5-7B': (383.0, 24.17, 7795.6, 433.1, 8635.8, 4.4, 95.0), 'LLaVA-NeXT-7B': (383.0, 24.17, 8375.4, 465.3, 9247.8, 4.1, 95.0),
    'Molmo-7B-D': (500.0, 0.0, 8914.3, 495.2, 9909.6, 5.0, 95.0), 'IDEFICS-9B': (335.0, 16.12, 34.6, 423.4, 809.2, 41.4, 47.7),
    'Llama-3.2-11B-Vision': (3290.1, 100.74, 866.2, 592.5, 4849.6, 67.8, 87.8), 'Idefics2-8B': (675.3, 95.13, 913.4, 456.7, 2140.5, 31.5, 78.7),
    'ViLT-B/32': (0.0, 0.0, 25.4, 5.7, 31.1, 0.0, 81.8), 'Qwen2-VL-7B': (1555.8, 22.82, 3649.8, 456.2, 5684.5, 27.4, 92.0),
    'Qwen2.5-VL-7B': (1557.4, 22.82, 3649.8, 456.2, 5686.2, 27.4, 92.0), 'InternVL3-8B': (726.5, 14.1, 3927.4, 490.9, 5158.9, 14.1, 90.5),
    'Pixtral-12B': (4954.7, 257.79, 108687.5, 849.1, 114749.1, 4.3, 99.3), 'DeepSeek-VL2-Small': (690.5, 19.88, 3563.6, 156.4, 4430.4, 15.6, 96.5),
    'Chameleon-7B': (0.0, 0.0, 14441.4, 451.3, 14892.7, 0.0, 97.0), 'Janus-Pro-7B': (397.0, 24.17, 8132.9, 451.8, 9005.8, 4.4, 95.0),
}
TABLE3_COLUMNS = ['encoder_g', 'fusion_g', 'llm_visual_g', 'llm_text_g', 'total_g', 'vision_compute_share_pct', 'vision_attributable_pct']

reproduction_rows = []
for name, published_row in PUBLISHED_TABLE3.items():
    computed = forward_cost(published_inputs[name])
    for column, published_value in zip(TABLE3_COLUMNS, published_row):
        reproduction_rows.append({'model': name, 'column': column, 'published': published_value,
                                  'reproduced': computed[column], 'difference': computed[column] - published_value})
reproduction = pd.DataFrame(reproduction_rows)
worst_cells = reproduction.reindex(reproduction['difference'].abs().sort_values(ascending=False).index).head(8)
print('Largest reproduction differences (printed values carry 1-2 decimals):')
worst_cells

Largest reproduction differences (printed values carry 1-2 decimals):


,model,column,published,reproduced,difference
87,ViLT-B/32,llm_text_g,5.7,5.651104,-0.048896
25,BLIP-large,total_g,392.4,392.351512,-0.048488
6,CLIP-ViT-L/14,vision_attributable_pct,95.4,95.351517,-0.048483
5,CLIP-ViT-L/14,vision_compute_share_pct,95.4,95.351517,-0.048483
100,Qwen2.5-VL-7B,llm_visual_g,3649.8,3649.752007,-0.047993
93,Qwen2-VL-7B,llm_visual_g,3649.8,3649.752007,-0.047993
82,Idefics2-8B,vision_compute_share_pct,31.5,31.547260,0.047260
95,Qwen2-VL-7B,total_g,5684.5,5684.547125,0.047125


In [4]:
# Acceptance: every published cell is matched to within one unit of its last printed digit.
tolerance = reproduction['published'].map(lambda value: 0.051 if round(value, 1) == value else 0.0051)
failures = reproduction[reproduction['difference'].abs() > tolerance + 1e-9]
print(f'{len(reproduction)} cells checked; {len(failures)} outside tolerance.')
failures

140 cells checked; 1 outside tolerance.


,model,column,published,reproduced,difference
113,Pixtral-12B,fusion_g,257.79,257.781924,-0.008076


In [5]:
revised_rows = []
for name in COMPUTE_SPECS:
    before, after = forward_cost(published_inputs[name]), forward_cost(revised_inputs[name])
    revised_rows.append({'model': name, **{column: after[column] for column in TABLE3_COLUMNS},
                         'published_total_g': PUBLISHED_TABLE3[name][4], 'published_C_V_pct': PUBLISHED_TABLE3[name][5],
                         'published_vision_attributable_pct': PUBLISHED_TABLE3[name][6],
                         'total_change_g': after['total_g'] - before['total_g']})
table3_revised = pd.DataFrame(revised_rows).set_index('model')
table3_revised.to_csv(MEASUREMENT_DIR / 'nb25_table3_revised.csv')
changed = table3_revised[table3_revised['total_change_g'].abs() > 0.05]
print('Rows whose total changes by more than 0.05 GFLOPs:')
changed.round(2)

Rows whose total changes by more than 0.05 GFLOPs:


,encoder_g,fusion_g,llm_visual_g,llm_text_g,total_g,vision_compute_share_pct,vision_attributable_pct,published_total_g,published_C_V_pct,published_vision_attributable_pct,total_change_g
model,,,,,,,,,,,
IDEFICS-9B,335.04,16.12,34.63,515.22,901.00,37.18,42.82,809.2,41.4,47.7,91.81
Qwen2-VL-7B,1464.46,22.82,3649.75,456.22,5593.26,26.18,91.84,5684.5,27.4,92.0,-91.29
Qwen2.5-VL-7B,1466.08,22.82,3649.75,456.22,5594.88,26.20,91.85,5686.2,27.4,92.0,-91.29


**Reading the changes.** IDEFICS-9B's text-token cost now includes the 1,434.5 M cross-attention parameters (they act on the text stream: Q, O and the gated MLP), exactly as Llama-3.2-Vision's already did. Qwen2-VL / Qwen2.5-VL no longer cost the 44.6 M merger twice (once inside the encoder over 1,024 patches, once as fusion over 256 tokens). InternVL3, Chameleon and Janus-Pro compute is unchanged: their parameter corrections were double counts or idle generation modules that the compute protocol never priced twice. BLIP-2 (+24,576 query-token parameters) and ViLT (+1,536 LayerNorm parameters) change by < 0.01 GFLOPs.

---
## 3. Table 7 and ρ(V%, C_V%) over the 19 separable models

In [6]:
PUBLISHED_TABLE7 = {  # V%, C_V%, ratio as printed
    'Llama-3.2-11B-Vision': (8.1, 67.8, 8.4), 'Idefics2-8B': (5.0, 31.5, 6.3), 'DeepSeek-VL2-Small': (2.7, 15.6, 5.8),
    'IDEFICS-9B': (8.4, 41.4, 4.9), 'InternVL3-8B': (3.6, 14.1, 3.9), 'Qwen2-VL-7B': (8.1, 27.4, 3.4), 'Qwen2.5-VL-7B': (8.1, 27.4, 3.4),
    'InstructBLIP-Vicuna-7B': (12.5, 37.3, 3.0), 'SigLIP2-So400m/14': (37.7, 93.8, 2.5), 'BLIP-2-OPT-2.7B': (26.3, 60.0, 2.3),
    'SigLIP-L/16': (48.5, 94.8, 2.0), 'BLIP-large': (68.0, 97.7, 1.4), 'Pixtral-12B': (3.2, 4.3, 1.4), 'CLIP-ViT-L/14': (70.9, 95.4, 1.3),
    'LLaVA-1.5-7B': (4.3, 4.4, 1.0), 'LLaVA-NeXT-7B': (4.0, 4.1, 1.0), 'Molmo-7B-D': (5.0, 5.0, 1.0), 'Janus-Pro-7B': (5.3, 4.4, 0.8),
    'ViLT-B/32': (22.2, 0.0, 0.0),
}
NB24_NAME = {'Llama-3.2-11B-Vision': 'Llama-3.2-11B-Vision-Instruct', 'Qwen2-VL-7B': 'Qwen2-VL-7B-Instruct', 'Qwen2.5-VL-7B': 'Qwen2.5-VL-7B-Instruct'}
components_revised = pd.read_csv(MEASUREMENT_DIR / 'nb24_parameter_components_revised.csv').set_index('model')

table7_rows = []
for name, (published_v, published_cv, published_ratio) in PUBLISHED_TABLE7.items():
    component_row = components_revised.loc[NB24_NAME.get(name, name)]
    vision_share = 100 * component_row['V_params'] / component_row['total_params']
    compute_share = table3_revised.loc[name, 'vision_compute_share_pct']
    table7_rows.append({'model': name, 'V_pct': vision_share, 'C_V_pct': compute_share,
                        'ratio': compute_share / vision_share,
                        'published_V_pct': published_v, 'published_C_V_pct': published_cv, 'published_ratio': published_ratio})
table7_revised = pd.DataFrame(table7_rows).sort_values('ratio', ascending=False).set_index('model')
table7_revised.to_csv(MEASUREMENT_DIR / 'nb25_table7_revised.csv')
table7_revised.round(2)

,V_pct,C_V_pct,ratio,published_V_pct,published_C_V_pct,published_ratio
model,,,,,,
Llama-3.2-11B-Vision,8.09,67.84,8.38,8.1,67.8,8.4
Idefics2-8B,4.97,31.55,6.34,5.0,31.5,6.3
DeepSeek-VL2-Small,2.65,15.58,5.88,2.7,15.6,5.8
IDEFICS-9B,7.06,37.18,5.26,8.4,41.4,4.9
InternVL3-8B,3.83,14.08,3.68,3.6,14.1,3.9
Qwen2-VL-7B,7.61,26.18,3.44,8.1,27.4,3.4
Qwen2.5-VL-7B,7.62,26.20,3.44,8.1,27.4,3.4
InstructBLIP-Vicuna-7B,12.46,37.31,2.99,12.5,37.3,3.0
SigLIP2-So400m/14,37.70,93.83,2.49,37.7,93.8,2.5


In [7]:
def correlation_report(vision_shares, compute_shares, label):
    """Spearman rho, permutation p (Monte Carlo, n = 19) and 95% Fisher interval."""
    test = spearman_permutation_test(vision_shares, compute_shares)
    interval = fisher_interval(test['rho'], test['sample_size'])
    print(f"{label}: rho = {test['rho']:+.3f}, n = {test['sample_size']}, permutation p = {test['p_value']:.4f} "
          f"({'exact' if test['exact'] else '20,000 draws'}), 95% Fisher interval [{interval[0]:+.2f}, {interval[1]:+.2f}]")
    return test, interval

rounded_correlation = correlation_report(table7_revised['published_V_pct'], table7_revised['published_C_V_pct'],
                                         'Published Table 7 values as printed (rounded, ties)')
archived_vision_share = {}
for name in table7_revised.index:
    stem, *_unused, encoder_key, _language_keys = COMPUTE_SPECS[name]
    archived = load_record(ARCHIVE_DIR, stem)
    vision_keys = ['vision_und', 'vision_gen'] if 'vision_und' in archived else ['vision']
    archived_total = sum((archived[key] or {}).get('params_total') or 0 for key in archived if key != 'vision' or 'vision_und' not in archived)
    archived_vision_share[name] = 100 * sum(archived[key]['params_total'] for key in vision_keys) / archived_total
published_compute_share = {name: forward_cost(published_inputs[name])['vision_compute_share_pct'] for name in table7_revised.index}
published_correlation = correlation_report(pd.Series(archived_vision_share), pd.Series(published_compute_share),
                                           'Published inputs, unrounded (reproduction; manuscript reports +0.72, p = 0.001, [+0.39, +0.88])')
revised_correlation = correlation_report(table7_revised['V_pct'], table7_revised['C_V_pct'], 'A13-revised inputs')

Published Table 7 values as printed (rounded, ties): rho = +0.734, n = 19, permutation p = 0.0006 (20,000 draws), 95% Fisher interval [+0.42, +0.89]
Published inputs, unrounded (reproduction; manuscript reports +0.72, p = 0.001, [+0.39, +0.88]): rho = +0.716, n = 19, permutation p = 0.0011 (20,000 draws), 95% Fisher interval [+0.39, +0.88]
A13-revised inputs: rho = +0.733, n = 19, permutation p = 0.0008 (20,000 draws), 95% Fisher interval [+0.42, +0.89]


---
## 4. Table 8 — the 28-test correlation grid

The grid uses the manuscript's own inputs (Tables 1, 4, 5, 6). A13 touches two predictors: **trainable share** (Llama-3.2-Vision 0.3 → 24.7 %) and **log total parameters** (Qwen2-VL, Qwen2.5-VL, InternVL3, Janus-Pro totals). The trainable values reproduce the published grid only if Idefics2 ranks just below Molmo (both print as '95' in Table 5); 94.9 is used — any value between 2.3 and 95.0 gives identical ranks. That Table 5 / Table 8 inconsistency is flagged for reviewer item A2.

Three runs isolate the two effects: **(a)** published inputs with the published method (check), **(b)** published inputs with exact enumeration for every test (method effect only), **(c)** A13-revised inputs with exact enumeration (A13 effect = c − b).

In [8]:
BENCHMARKS = ['ai2d', 'chartqa', 'docvqa', 'mathvista', 'mmbench_en', 'mmmu', 'seedbench']
PREDICTORS = ['trainable_pct', 'log tokens_default', 'log tokens_dynamic', 'log total_params']
# model: year, tokens default, tokens dynamic, trainable % (grid), total M (published), total M (revised), scores
GRID_MODELS = {
    'InstructBLIP-Vicuna-7B': (2023, 32, 32, 2.3, 7913.7, 7913.7, {'seedbench': 53.4}),
    'LLaVA-1.5-7B': (2023, 576, 576, 95.6, 7063.4, 7063.4, {'mmbench_en': 64.3, 'seedbench': 58.6}),
    'LLaVA-NeXT-7B': (2024, 576, 2880, 95.9, 7566.7, 7566.7, {'ai2d': 66.6, 'chartqa': 54.8, 'docvqa': 74.4, 'mmbench_en': 67.4, 'mmmu': 35.1}),
    'Idefics2-8B': (2024, 64, 64, 94.9, 8402.8, 8402.8, {'ai2d': 72.3, 'docvqa': 74.0, 'mathvista': 51.4, 'mmbench_en': 75.7, 'mmmu': 43.5}),
    'Qwen2-VL-7B': (2024, 256, 1024, 100.0, 8336.0, 8291.4, {'ai2d': 83.0, 'chartqa': 83.0, 'docvqa': 94.5, 'mathvista': 58.2, 'mmbench_en': 81.0, 'mmmu': 54.1, 'seedbench': 76.0}),
    'Qwen2.5-VL-7B': (2025, 256, 1024, 100.0, 8336.7, 8292.2, {'ai2d': 83.9, 'chartqa': 87.3, 'docvqa': 95.7, 'mathvista': 68.2, 'mmbench_en': 83.5, 'mmmu': 58.6, 'seedbench': 77.0}),
    'InternVL3-8B': (2025, 256, 3072, 100.0, 8488.0, 7944.4, {'ai2d': 85.2, 'chartqa': 86.6, 'docvqa': 92.7, 'mathvista': 71.6, 'mmbench_en': 83.4, 'mmmu': 62.7}),
    'Pixtral-12B': (2024, 4096, 4096, 100.0, 12682.7, 12682.7, {'chartqa': 81.8, 'docvqa': 90.7, 'mathvista': 58.0, 'mmbench_en': 72.7, 'mmmu': 52.5}),
    'Llama-3.2-11B-Vision': (2024, 1601, 1601, 0.3, 10670.2, 10670.2, {'ai2d': 91.1, 'chartqa': 83.4, 'docvqa': 88.4, 'mathvista': 51.5, 'mmmu': 50.7}),
    'DeepSeek-VL2-Small': (2024, 729, 729, 97.3, 16148.3, 16148.3, {'ai2d': 83.8, 'chartqa': 84.5, 'docvqa': 92.3, 'mathvista': 62.8, 'mmmu': 48.0}),
    'Janus-Pro-7B': (2025, 576, 576, 99.1, 7319.6, 7420.4, {'mmbench_en': 79.2, 'mmmu': 41.0, 'seedbench': 72.1}),
    'Molmo-7B-D': (2024, 576, 2560, 95.0, 8021.0, 8021.0, {'chartqa': 84.1, 'docvqa': 92.2, 'mathvista': 51.6, 'mmmu': 45.3}),
}
REVISED_TRAINABLE = {'Llama-3.2-11B-Vision': round(100 * components_revised.loc['Llama-3.2-11B-Vision-Instruct', 'trainable_params']
                                                       / components_revised.loc['Llama-3.2-11B-Vision-Instruct', 'total_params'], 2)}
print('Revised trainable inputs:', REVISED_TRAINABLE)

def grid_columns(benchmark, predictor, revised):
    """Structural values, scores and years for the models scored on one benchmark."""
    structural, scores, years = [], [], []
    for name, (year, tokens_default, tokens_dynamic, trainable, total_published, total_revised, model_scores) in GRID_MODELS.items():
        if benchmark not in model_scores:
            continue
        trainable_value = REVISED_TRAINABLE.get(name, trainable) if revised else trainable
        total_value = total_revised if revised else total_published
        structural.append({'trainable_pct': trainable_value, 'log tokens_default': math.log(tokens_default),
                           'log tokens_dynamic': math.log(tokens_dynamic), 'log total_params': math.log(total_value)}[predictor])
        scores.append(model_scores[benchmark]); years.append(year)
    return structural, scores, years

def run_grid(revised):
    """All 28 tests with exact enumeration, plus BH within the marginal and the partial families."""
    rows = []
    for benchmark in BENCHMARKS:
        for predictor in PREDICTORS:
            result = spearman_with_partial(*grid_columns(benchmark, predictor, revised))
            interval = fisher_interval(result['rho'], result['sample_size']) if result['sample_size'] >= 8 else None
            rows.append({'benchmark': benchmark, 'predictor': predictor, 'n': result['sample_size'],
                         'rho': result['rho'], 'p_perm': result['p_value'],
                         'ci_low': interval[0] if interval else np.nan, 'ci_high': interval[1] if interval else np.nan,
                         'partial_rho_year': result['partial_rho'], 'partial_p': result['partial_p_value']})
    grid = pd.DataFrame(rows)
    grid['p_bh'] = benjamini_hochberg(grid['p_perm'])
    grid['partial_p_bh'] = benjamini_hochberg(grid['partial_p'])
    return grid

grid_method_only = run_grid(revised=False)
grid_revised = run_grid(revised=True)
print('done')

Revised trainable inputs: {'Llama-3.2-11B-Vision': np.float64(24.74)}


done


In [9]:
PUBLISHED_TABLE8 = [  # rho, p_perm, p_bh, partial rho, partial p_bh — in BENCHMARKS x PREDICTORS order
    (0.074, 0.8929, 0.9616, -0.357, 0.7831), (0.371, 0.4333, 0.8088, 0.633, 0.5244), (0.324, 0.4817, 0.843, 0.167, 0.9511), (0.643, 0.1389, 0.4939, 0.73, 0.3394),
    (0.228, 0.5833, 0.8698, -0.322, 0.7831), (-0.466, 0.2435, 0.6198, 0.049, 0.9511), (-0.395, 0.3331, 0.7198, -0.678, 0.3394), (0.262, 0.5364, 0.8698, 0.4, 0.7831),
    (0.749, 0.0283, 0.3108, 0.626, 0.3394), (-0.196, 0.6213, 0.8698, 0.094, 0.9511), (-0.025, 0.956, 0.9814, -0.158, 0.9511), (0.017, 0.9814, 0.9814, 0.021, 0.9537),
    (0.786, 0.031, 0.3108, 0.689, 0.3394), (-0.098, 0.8324, 0.9616, 0.323, 0.7831), (0.216, 0.6085, 0.8698, 0.112, 0.9511), (0.095, 0.8401, 0.9616, 0.293, 0.7831),
    (0.66, 0.0845, 0.3943, 0.485, 0.6883), (-0.576, 0.1411, 0.4939, -0.675, 0.3394), (0.12, 0.7831, 0.9616, -0.046, 0.9511), (0.381, 0.3599, 0.7198, 0.284, 0.7831),
    (0.644, 0.0527, 0.3689, 0.581, 0.3394), (-0.155, 0.6755, 0.9007, -0.04, 0.9511), (0.34, 0.3363, 0.7198, 0.391, 0.6883), (0.479, 0.1649, 0.513, 0.629, 0.3394),
    (0.975, 0.0333, 0.3108, 0.95, 0.3394), (0.105, 0.8667, 0.9616, -0.156, 0.9511), (0.949, 0.0667, 0.3735, 0.98, 0.3394), (0.7, 0.2333, 0.6198, 0.603, 0.7831),
]
published_grid = pd.DataFrame(PUBLISHED_TABLE8, columns=['rho', 'p_perm', 'p_bh', 'partial_rho_year', 'partial_p_bh'])
check = pd.DataFrame({
    'max |rho - published|': [ (grid_method_only['rho'] - published_grid['rho']).abs().max() ],
    'max |partial rho - published|': [ (grid_method_only['partial_rho_year'] - published_grid['partial_rho_year']).abs().max() ],
    'max |p - published|, n<=8 (exact both)': [ (grid_method_only['p_perm'] - published_grid['p_perm'])[grid_method_only['n'] <= 8].abs().max() ],
    'max |p - published|, n=9,10 (published used 20,000 draws)': [ (grid_method_only['p_perm'] - published_grid['p_perm'])[grid_method_only['n'] > 8].abs().max() ],
    'max |partial BH - published|': [ (grid_method_only['partial_p_bh'] - published_grid['partial_p_bh']).abs().max() ],
    'min BH (marginal / partial)': [ f"{grid_method_only['p_bh'].min():.4f} / {grid_method_only['partial_p_bh'].min():.4f}" ],
}).T
check.columns = ['run (b): published inputs, exact enumeration']
check

,"run (b): published inputs, exact enumeration"
max |rho - published|,0.000491
max |partial rho - published|,0.000494
"max |p - published|, n<=8 (exact both)",0.000048
"max |p - published|, n=9,10 (published used 20,000 draws)",0.006154
max |partial BH - published|,0.0144
min BH (marginal / partial),0.3111 / 0.3296


In [10]:
# A13 effect: compare run (c) with run (b) cell by cell.
compare = grid_revised.copy()
for column in ['rho', 'p_perm', 'p_bh', 'partial_rho_year', 'partial_p', 'partial_p_bh']:
    compare[f'{column}_before'] = grid_method_only[column]
affected = compare[(compare['rho'] - compare['rho_before']).abs() > 1e-9]
print('Tests whose rho changes because of A13 (all others keep identical rho and partial rho):')
print(affected[['benchmark','predictor','n','rho_before','rho','p_perm_before','p_perm','partial_rho_year_before','partial_rho_year']].round(4).to_string(index=False))
print()
trainable_rows = compare[compare['predictor'] == 'trainable_pct']
print('trainable_pct rows identical after Llama 0.3 -> 24.7 %:', bool(((trainable_rows['rho'] - trainable_rows['rho_before']).abs() < 1e-12).all()))
print(f"Minimum BH-adjusted p after A13: marginal {grid_revised['p_bh'].min():.4f}, year-partial {grid_revised['partial_p_bh'].min():.4f}")
grid_revised.to_csv(MEASUREMENT_DIR / 'nb25_table8_revised.csv', index=False)
grid_method_only.to_csv(MEASUREMENT_DIR / 'nb25_table8_published_inputs_exact.csv', index=False)

Tests whose rho changes because of A13 (all others keep identical rho and partial rho):
 benchmark        predictor  n  rho_before     rho  p_perm_before  p_perm  partial_rho_year_before  partial_rho_year
      ai2d log total_params  7      0.6429  0.3571         0.1389  0.4444                   0.7302            0.6072
   chartqa log total_params  8      0.2619  0.1429         0.5364  0.7520                   0.4001            0.5262
    docvqa log total_params  9      0.0167 -0.0667         0.9816  0.8801                   0.0213            0.1694
 mathvista log total_params  8      0.0952 -0.2857         0.8401  0.5008                   0.2933            0.1684
mmbench_en log total_params  8      0.3810  0.3095         0.3599  0.4618                   0.2844            0.4131
      mmmu log total_params 10      0.4788  0.2848         0.1663  0.4271                   0.6289            0.5552

trainable_pct rows identical after Llama 0.3 -> 24.7 %: True
Minimum BH-adjusted p after A13

---
## 5. Section 5.8 — composite score, score per TFLOP, Pareto frontier

In [11]:
EFFICIENCY_NAMES = ['InstructBLIP-Vicuna-7B', 'Idefics2-8B', 'DeepSeek-VL2-Small', 'Llama-3.2-11B-Vision', 'InternVL3-8B', 'Qwen2-VL-7B',
                    'Qwen2.5-VL-7B', 'LLaVA-1.5-7B', 'Janus-Pro-7B', 'LLaVA-NeXT-7B', 'Molmo-7B-D', 'Pixtral-12B']

def pareto_flags(scores, compute):
    """True where no other model has score >= and compute <= with at least one strict."""
    flags = []
    for index, (score, cost) in enumerate(zip(scores, compute)):
        dominated = any((other_score >= score and other_cost <= cost and (other_score > score or other_cost < cost))
                        for other_index, (other_score, other_cost) in enumerate(zip(scores, compute)) if other_index != index)
        flags.append(not dominated)
    return flags

def efficiency_table(total_column, benchmark=None):
    """Composite (or single-benchmark) score, TFLOPs, score/TFLOP and Pareto flag."""
    rows = []
    for name in EFFICIENCY_NAMES:
        model_scores = GRID_MODELS[name][6]
        if benchmark is not None and benchmark not in model_scores:
            continue
        values = [model_scores[benchmark]] if benchmark else list(model_scores.values())
        teraflops = total_column[name] / 1000
        rows.append({'model': name, 'n_benchmarks': len(values), 'score': np.mean(values), 'TFLOPs': teraflops,
                     'score_per_TFLOP': np.mean(values) / teraflops})
    table = pd.DataFrame(rows)
    table['pareto'] = pareto_flags(table['score'], table['TFLOPs'])
    return table.sort_values('TFLOPs').set_index('model')

published_totals = {name: PUBLISHED_TABLE3[name][4] for name in PUBLISHED_TABLE3}
revised_totals = table3_revised['total_g'].to_dict()
for label, benchmark in (('composite', None), ('MMMU', 'mmmu'), ('DocVQA', 'docvqa')):
    before, after = efficiency_table(published_totals, benchmark), efficiency_table(revised_totals, benchmark)
    merged = after.join(before[['TFLOPs', 'score_per_TFLOP', 'pareto']], rsuffix='_published')
    merged.to_csv(MEASUREMENT_DIR / f'nb25_efficiency_{label.lower()}_revised.csv')
    print(f'--- {label}: Pareto set published {sorted(before.index[before.pareto])}')
    print(f'    {" " * len(label)}  Pareto set revised   {sorted(after.index[after.pareto])}')
    display(merged.round(2))

--- composite: Pareto set published ['DeepSeek-VL2-Small', 'Idefics2-8B', 'InstructBLIP-Vicuna-7B', 'InternVL3-8B']
               Pareto set revised   ['DeepSeek-VL2-Small', 'Idefics2-8B', 'InstructBLIP-Vicuna-7B', 'InternVL3-8B']


,n_benchmarks,score,TFLOPs,score_per_TFLOP,pareto,TFLOPs_published,score_per_TFLOP_published,pareto_published
model,,,,,,,,
InstructBLIP-Vicuna-7B,1,53.40,1.40,38.19,True,1.40,38.19,True
Idefics2-8B,5,63.38,2.14,29.61,True,2.14,29.61,True
DeepSeek-VL2-Small,5,74.28,4.43,16.77,True,4.43,16.77,True
Llama-3.2-11B-Vision,5,73.02,4.85,15.06,False,4.85,15.06,False
InternVL3-8B,6,80.37,5.16,15.58,True,5.16,15.58,True
Qwen2-VL-7B,7,75.69,5.59,13.53,False,5.68,13.31,False
Qwen2.5-VL-7B,7,79.17,5.59,14.15,False,5.69,13.92,False
LLaVA-1.5-7B,2,61.45,8.64,7.12,False,8.64,7.12,False
Janus-Pro-7B,3,64.10,9.01,7.12,False,9.01,7.12,False


--- MMMU: Pareto set published ['DeepSeek-VL2-Small', 'Idefics2-8B', 'InternVL3-8B', 'Llama-3.2-11B-Vision']
          Pareto set revised   ['DeepSeek-VL2-Small', 'Idefics2-8B', 'InternVL3-8B', 'Llama-3.2-11B-Vision']


,n_benchmarks,score,TFLOPs,score_per_TFLOP,pareto,TFLOPs_published,score_per_TFLOP_published,pareto_published
model,,,,,,,,
Idefics2-8B,1,43.5,2.14,20.32,True,2.14,20.32,True
DeepSeek-VL2-Small,1,48.0,4.43,10.83,True,4.43,10.83,True
Llama-3.2-11B-Vision,1,50.7,4.85,10.45,True,4.85,10.45,True
InternVL3-8B,1,62.7,5.16,12.15,True,5.16,12.15,True
Qwen2-VL-7B,1,54.1,5.59,9.67,False,5.68,9.52,False
Qwen2.5-VL-7B,1,58.6,5.59,10.47,False,5.69,10.31,False
Janus-Pro-7B,1,41.0,9.01,4.55,False,9.01,4.55,False
LLaVA-NeXT-7B,1,35.1,9.25,3.80,False,9.25,3.80,False
Molmo-7B-D,1,45.3,9.91,4.57,False,9.91,4.57,False


--- DocVQA: Pareto set published ['DeepSeek-VL2-Small', 'Idefics2-8B', 'InternVL3-8B', 'Qwen2-VL-7B', 'Qwen2.5-VL-7B']
            Pareto set revised   ['DeepSeek-VL2-Small', 'Idefics2-8B', 'InternVL3-8B', 'Qwen2-VL-7B', 'Qwen2.5-VL-7B']


,n_benchmarks,score,TFLOPs,score_per_TFLOP,pareto,TFLOPs_published,score_per_TFLOP_published,pareto_published
model,,,,,,,,
Idefics2-8B,1,74.0,2.14,34.57,True,2.14,34.57,True
DeepSeek-VL2-Small,1,92.3,4.43,20.83,True,4.43,20.83,True
Llama-3.2-11B-Vision,1,88.4,4.85,18.23,False,4.85,18.23,False
InternVL3-8B,1,92.7,5.16,17.97,True,5.16,17.97,True
Qwen2-VL-7B,1,94.5,5.59,16.90,True,5.68,16.62,True
Qwen2.5-VL-7B,1,95.7,5.59,17.10,True,5.69,16.83,True
LLaVA-NeXT-7B,1,74.4,9.25,8.05,False,9.25,8.05,False
Molmo-7B-D,1,92.2,9.91,9.30,False,9.91,9.30,False
Pixtral-12B,1,90.7,114.75,0.79,False,114.75,0.79,False


---
## 6. Summary of A13 downstream effects

In [12]:
summary_lines = [
    f"Table 3 rows changed: {', '.join(changed.index)}",
    f"IDEFICS-9B total {PUBLISHED_TABLE3['IDEFICS-9B'][4]} -> {table3_revised.loc['IDEFICS-9B','total_g']:.1f} GFLOPs; "
    f"C_V% {PUBLISHED_TABLE3['IDEFICS-9B'][5]} -> {table3_revised.loc['IDEFICS-9B','vision_compute_share_pct']:.1f}; "
    f"vision-attributable {PUBLISHED_TABLE3['IDEFICS-9B'][6]} -> {table3_revised.loc['IDEFICS-9B','vision_attributable_pct']:.1f}%",
    f"rho(V%, C_V%): published inputs {published_correlation[0]['rho']:+.3f} -> revised {revised_correlation[0]['rho']:+.3f} "
    f"[{revised_correlation[1][0]:+.2f}, {revised_correlation[1][1]:+.2f}], p = {revised_correlation[0]['p_value']:.4f}",
    f"Table 8: {len(affected)} of 56 correlations change (log total_params only); no test survives BH "
    f"(min adjusted p marginal {grid_revised['p_bh'].min():.3f}, partial {grid_revised['partial_p_bh'].min():.3f})",
]
print('\n'.join(summary_lines))

Table 3 rows changed: IDEFICS-9B, Qwen2-VL-7B, Qwen2.5-VL-7B
IDEFICS-9B total 809.2 -> 901.0 GFLOPs; C_V% 41.4 -> 37.2; vision-attributable 47.7 -> 42.8%
rho(V%, C_V%): published inputs +0.716 -> revised +0.733 [+0.42, +0.89], p = 0.0008
Table 8: 6 of 56 correlations change (log total_params only); no test survives BH (min adjusted p marginal 0.311, partial 0.416)
